# PCB 부품 검출 - Colab 학습 (base / rot)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hi839034-lgtm/pcb-/blob/main/pcb_eval/colab/colab_train.ipynb)

**준비물은 원본 데이터 `pcb-component-detection-DatasetNinja.tar` 하나뿐입니다.** (코드는 GitHub 에서 자동으로 받고, 학습용 분할은 노트북이 직접 만듭니다. 드라이브 검색/zip 업로드가 필요 없습니다.)

1. 런타임 > 런타임 유형 변경 > **T4 GPU**
2. 아래 셀을 **위에서부터 순서대로** 실행 (셀 1 → 2 → 3 → 4(base) → 5 → 6(rot) → 7)
3. 학습 하나가 끝날 때마다 바로 `best.pt` 를 내려받기 (런타임이 끊기면 `/content` 가 지워집니다)

※ base 와 rot 은 **회전 증강(degrees, flipud)과 name 만 다릅니다.** 나머지 설정은 같아야 비교가 공정합니다.

## 1. 환경 + 코드 받기

In [ ]:
import torch
!pip -q install ultralytics
!rm -rf /content/pcb && git clone -q https://github.com/hi839034-lgtm/pcb- /content/pcb
assert torch.cuda.is_available(), "GPU 가 꺼져 있음: 런타임 > 런타임 유형 변경 > T4 GPU"
print('GPU:', torch.cuda.get_device_name(0))
!ls /content/pcb/pcb_eval/src

## 2. 원본 tar 위치 확인
- 이미 `/content/` 에 올려 둔 tar 가 있으면 자동으로 사용합니다.
- 없으면 파일 선택 창이 뜹니다 → `pcb-component-detection-DatasetNinja.tar` (약 188MB) 선택 후 **업로드가 끝날 때까지 기다리세요.**

In [ ]:
import os, tarfile
NAME = 'pcb-component-detection-DatasetNinja.tar'
TAR = f'/content/{NAME}'
if not os.path.exists(TAR):
    from google.colab import files
    up = files.upload()
    TAR = '/content/' + next(iter(up))
print('tar:', TAR, round(os.path.getsize(TAR) / 1e6), 'MB')      # 약 188MB
with tarfile.open(TAR) as t:                                       # 끝까지 읽어 파일이 온전한지 검사
    n = len(t.getnames())
print('항목 수:', n)
assert n == 2823, "tar 가 불완전함(업로드가 중간에 끊김) - 다시 올리세요" 

## 3. 학습용 분할 만들기 (영상 단위, PC 와 동일한 test 가 나와야 함)

In [ ]:
!rm -rf /content/raw_dataset /content/pcb_yolo
!mkdir -p /content/raw_dataset
!tar -xf "$TAR" -C /content/raw_dataset
!python /content/pcb/pcb_eval/src/prepare_dataset.py --raw /content/raw_dataset --out /content/pcb_yolo

import json, collections
m = json.load(open('/content/pcb_yolo/split_manifest.json'))
cnt = collections.Counter(r['split'] for r in m)
test_videos = sorted({r['video'] for r in m if r['split'] == 'test'})
EXPECTED_TEST = ['VID20210601143927', 'VID20210601144135', 'VID20210601144202', 'VID20210601144238', 'VID202106071438551', 'VID202106071439081', 'VID202106071442011']
print(dict(cnt))
assert dict(cnt) == {'train': 921, 'val': 225, 'test': 50}, "분할 개수가 PC 와 다름"
assert test_videos == EXPECTED_TEST, "test 영상이 PC 와 다름 - 평가가 무효가 되므로 중단"
print('OK: PC 와 같은 분할 (test 영상 7개 일치)')

## 4. base 학습 - 회전 증강 **없음** (보완 전)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/pcb_yolo/data.yaml',
            project='/content/pcb_runs', name='base', exist_ok=True,
            epochs=100, imgsz=640, batch=16, seed=0, workers=2,
            degrees=0, flipud=0.0)

In [ ]:
# base 내려받기 (학습이 끝난 직후 실행)
import shutil
from google.colab import files
shutil.copy('/content/pcb_runs/base/weights/best.pt', '/content/base_best.pt')
files.download('/content/base_best.pt')

## 5. rot 학습 - 회전 증강 **있음** (보완 후)
런타임이 새로 시작됐다면 셀 1~3 을 먼저 다시 실행하세요.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/pcb_yolo/data.yaml',
            project='/content/pcb_runs', name='rot', exist_ok=True,
            epochs=100, imgsz=640, batch=16, seed=0, workers=2,
            degrees=30, flipud=0.5)

In [ ]:
# rot 내려받기
import shutil
from google.colab import files
shutil.copy('/content/pcb_runs/rot/weights/best.pt', '/content/rot_best.pt')
files.download('/content/rot_best.pt')

## 6. 확인 (val 기준, test 는 보지 않음)

In [ ]:
import os
from ultralytics import YOLO
for n in ['base', 'rot']:
    w = f'/content/pcb_runs/{n}/weights/best.pt'
    if os.path.exists(w):
        r = YOLO(w).val(data='/content/pcb_yolo/data.yaml', split='val', verbose=False)
        print(n, 'val mAP50 =', round(r.box.map50, 3), ' mAP50-95 =', round(r.box.map, 3))
    else:
        print(n, '결과 없음')

## 7. PC 에서 이어서
받은 `base_best.pt`, `rot_best.pt` 를 PC 의 `pcb_eval/runs/base/weights/best.pt`, `pcb_eval/runs/rot/weights/best.pt` 로 복사한 뒤
`pcb_eval/README.md` 의 2~4단계(캔버스 생성 → 웹캠 촬영 → 평가)를 진행합니다.